In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np

In this Notebook we will train a ConvNet to classify images from Cifar10 dataset
The Cifar10 dataset is included in torch-vision package 
In this image classification we will classify images from 10 classes given below:
plane, car , bird, cat deer, dog, frog, horse, ship, truck

By specifying train = True we will load the training data from the dataset and test dataset with train = False

In [2]:
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data.sampler import SubsetRandomSampler

# Number of subprocesses to use for data loading
num_workers = 1
# how many samples per batch to load 
batch_size = 10

#convert data to a normalized  torch.FloatTensor
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

#choose the training and test datasets
train_data = datasets.CIFAR10('data',train=True,download=True,transform=transform)
test_data = datasets.CIFAR10('data',train=False,download=True,transform=transform)

# prepare data loaders 
train_loader = torch.utils.data.DataLoader(train_data,batch_size=batch_size,num_workers=num_workers)
test_loader = torch.utils.data.DataLoader(test_data,batch_size=batch_size,num_workers=num_workers)




c:\Users\karti\anaconda3\envs\torchfix\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Augmentation means making something larger, better, or more varied by adding extra things to it.

In Machine Learning / Deep Learning, data augmentation means creating new training examples from existing data by applying small changes.

Example for images:

- Rotate image
- Flip image
- Zoom in/out
- Change brightness
- Crop image
- Add noise

So if you have 1 cat image, augmentation can create many slightly different cat images. This helps the model learn better and avoid overfitting.

Why we use augmentation:

Increases dataset size
Improves model accuracy
Reduces overfitting
Helps model handle real-world variations

In [3]:
images_batch , label_batch = next(iter(train_loader))

In [4]:
images_batch.shape, label_batch.shape

(torch.Size([10, 3, 32, 32]), torch.Size([10]))

# Defining the Architecture
```
Input Image
   ↓
Convolution Layer
   ↓
Activation Function (ReLU)
   ↓
Pooling Layer
   ↓
Convolution Layer
   ↓
Activation Function (ReLU)
   ↓
Pooling Layer
   ↓
Flatten Layer
   ↓
Fully Connected Layer
   ↓
Output Layer
```

In [6]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # convolutional layer (sees 32x32x3 image tensor)
        # torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
        self.conv1 = nn.Conv2d(3,16,3,padding=1)
        # convolutional layer (sees 16x16x16 tensor) -> calculation shown below
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        # convolutional layer(sees 8x8x32 tensor)
        self.conv3 = nn.Conv2d(32,64,3,padding=1)
        # max pooling layer
        self.pool = nn.MaxPool2d(2,2)
        # linear layer (64* 4 *4 -> 500)
        self.fc1 = nn.Linear(64 * 4 * 4, 500)
        # linear layer (500 -> 10)
        self.fc2 = nn.Linear(500,10) # output layer
        #dropout layer (p = 0.25)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        # add sequence of convolutional and max pooling layers
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # flatten layer
        x = x.view(-1, 64 * 4 * 4)

        # add dropout layer
        x = self.dropout(x)

        # add 1st hidden layer, with relu activation function
        x = F.relu(self.fc1(x))
        # add dropout layer
        x = self.dropout(x)

        # add 2nd hidden layer, with relu activation function
        x = self.fc2(x) # also output layer therfore no dropout and no relu

        return x
    

# create a complete CNN
model = Net()
print(model)



Net(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1024, out_features=500, bias=True)
  (fc2): Linear(in_features=500, out_features=10, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
)


Output volume can be calculated with below formula:

  - Input: n X n X nc
  - Filter: f X f X nc
  - Padding: p
  - Stride: s
  - Output: [((n+2p-f)/s)+1] X [((n+2p-f)/s)+1] X nc’ (height X width X no of output channels)
nc is the number of channels in the input and filter, while nc’ is the number of filters.

From the above structure you can see that height/width is getting reduced and number of channels are getting incresed.

Example calulating the output of first convolution + pooling layer operation -

Input image shape - 32(n) X 32(n) X 3(nc)

#### 1. ConVNet filter operation - self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)

Filter shape - 3 (f) X 3 (f) X 3(nc) Padding : P = 1 Stride : s = 1 (default value) output channels - 16 (kernel_size)

putting it in the formula given above -

[((n+2p-f)/s)+1] X [((n+2p-f)/s)+1] X nc’

[((32 + 2X1 - 3) / 1) + 1)] X [((32 + 2X1 - 3) / 1)) + 1)] X 16

output shape -> 32 X 32 X 16

#### 2. output of conv1 is passed through max pooling layer.
self.pool = nn.MaxPool2d(2, 2) -> filter of 2 X 2.

this will shrink the height & width by half , however no of channels will remain same.

input to the pooling layer - 32 X 32 X 16

output of the pooing layer - 32/2 X 32/2 X 16 -> 16 X 16 X 16

### Defining the learning rate, loss function and optimizer

In [9]:
learning_rate = 0.001

criterion = nn.CrossEntropyLoss() # As we know CrossEntropyLoss always uses Softmax therefore we didnt used softmax in above architecture

optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

### Start the training

In [11]:
total_step = len(train_loader)
num_epochs = 10

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1)% 2000 == 0 :
            print('Epoch [{}/{}], Step[{}/{}] ,Loss: {:.4f}'.format(epoch+1,num_epochs,i+1,total_step,loss.item()))

Epoch [1/10], Step[2000/5000] ,Loss: 0.9477
Epoch [1/10], Step[4000/5000] ,Loss: 0.7028
Epoch [2/10], Step[2000/5000] ,Loss: 0.6917
Epoch [2/10], Step[4000/5000] ,Loss: 0.8238
Epoch [3/10], Step[2000/5000] ,Loss: 1.0232
Epoch [3/10], Step[4000/5000] ,Loss: 0.4046
Epoch [4/10], Step[2000/5000] ,Loss: 0.6977
Epoch [4/10], Step[4000/5000] ,Loss: 0.5600
Epoch [5/10], Step[2000/5000] ,Loss: 0.6119
Epoch [5/10], Step[4000/5000] ,Loss: 0.4508
Epoch [6/10], Step[2000/5000] ,Loss: 0.3989
Epoch [6/10], Step[4000/5000] ,Loss: 0.2468
Epoch [7/10], Step[2000/5000] ,Loss: 0.3228
Epoch [7/10], Step[4000/5000] ,Loss: 0.3045
Epoch [8/10], Step[2000/5000] ,Loss: 0.3459
Epoch [8/10], Step[4000/5000] ,Loss: 0.5069
Epoch [9/10], Step[2000/5000] ,Loss: 0.6421
Epoch [9/10], Step[4000/5000] ,Loss: 0.4019
Epoch [10/10], Step[2000/5000] ,Loss: 0.3902
Epoch [10/10], Step[4000/5000] ,Loss: 0.3380


### Evaluating the model on Test Data

In [14]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for images , labels in test_loader:

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the model on the 1000 test images : {}%'.format(100*correct/total))

Accuracy of the model on the 1000 test images : 71.77%


### NOTE 
num_workers controls how data is loaded.
0 means main process loads data.
1 means one helper process loads data.

Normalize(0.5, 0.5) changes values from 0–1 to -1–1.
For your own dataset, calculate mean and std.
For pretrained ImageNet models, use ImageNet mean/std.

ToTensor does not destroy spatial structure.
It keeps image as Channels x Height x Width.
Flattening destroys spatial structure, not ToTensor.

Normalization helps training because values become centered and stable.

16, 32, 64 channels are design choices.
You can use other numbers, but next layers must match.

Conv2d is used because images are 2D.
Conv1d is for sequence/audio/text/time-series.
Conv3d is for video/3D medical scans/volumes.

Kernel size 3 means 3x3 filter.
It is popular because it is small, efficient, and powerful.

Three conv layers are used as a beginner-friendly balance.
More layers can improve learning but also increase training time and overfitting risk.